In [0]:
# Widgets
dbutils.widgets.text("landing_path", "")
dbutils.widgets.text("archive_path", "")
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("bronze_schema_name", "")

# Read values
landing_path = dbutils.widgets.get("landing_path").strip()
archive_path = dbutils.widgets.get("archive_path").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()
bronze_schema_name = dbutils.widgets.get("bronze_schema_name").strip()

# List files
files = dbutils.fs.ls(landing_path)

for f in files:
    # Read file
    cafe_sales_df = spark.read.csv(f.path, header=True, inferSchema=True)

    bronze_table_name = "cafe_sales_raw"
    full_table_name = f"{catalog_name}.{bronze_schema_name}.{bronze_table_name}"

    if spark.catalog.tableExists(full_table_name):
        spark.sql(f"TRUNCATE TABLE `{catalog_name}`.`{bronze_schema_name}`.`{bronze_table_name}`")
        cafe_sales_df.write.format("delta").mode("append").saveAsTable(full_table_name)
    else:
        cafe_sales_df.write.format("delta") \
            .option("delta.columnMapping.mode", "name") \
            .mode("overwrite") \
            .saveAsTable(full_table_name)

    # Move file to archive
    dbutils.fs.mv(f.path, archive_path + f.name)
